# 02 · Econometría — correlaciones, regresión vs Brent y proyección

> **Correlación no es causalidad.** Los modelos describen asociaciones anuales; no aíslan efectos de volumen, diferenciales de precio del crudo ecuatoriano, tipo de cambio de los socios ni política.

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = Path(os.getenv("APP_DATA_DIR", ROOT / "app" / "data"))   # snapshot gold exportado por el pipeline
data = {p.stem: pd.read_parquet(p) for p in DATA.glob("*.parquet")}
meta = json.loads((DATA / "metadata.json").read_text(encoding="utf-8")) if (DATA / "metadata.json").exists() else {}
print("Snapshot exportado:", meta.get("exported_at_utc"), "| tablas:", len(data))

COLORS = {"ECU": "#2a78d6", "PER": "#eb6834", "COL": "#1baf7a", "CHL": "#eda100", "LCN": "#8a8984"}
PCOLORS = {"oil": "#2a78d6", "shrimp": "#eb6834", "banana": "#1baf7a", "cocoa": "#eda100", "mining": "#e87ba4",
           "flowers": "#008300", "tuna": "#6250d6", "other": "#a3a29c"}
plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 11})
def shade_events(ax):
    for e in data["economic_events"].itertuples():
        ax.axvspan(pd.Timestamp(e.start_date).year - 0.5, pd.Timestamp(e.end_date).year + 0.5, color="grey", alpha=0.12)
dep, basket, div = data["oil_dependency"], data["export_basket"], data["diversification"]
ecu = dep[dep.iso3 == "ECU"].sort_values("year")
y0, y1 = int(ecu.year.min()), int(ecu.year.max())
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX

## 1. Matriz de correlaciones (variaciones anuales en logaritmos)

In [ ]:
br = data["oil_price_annual"].query("series == 'brent'")[["year", "avg_price_usd_bbl"]]
rc = data["regional_compare"].query("iso3 == 'ECU'").pivot_table(index="year", columns="indicator_code", values="value")
df = ecu.set_index("year")[["oil_exports_usd", "non_oil_exports_usd", "total_exports_usd"]].join(br.set_index("year")).join(rc[["NY.GDP.MKTP.CD"]])
dl = np.log(df).diff().dropna()
corr = dl.corr().round(2); display(corr)
print(f"Conclusión: Δln(Brent) se correlaciona {corr.loc['avg_price_usd_bbl','oil_exports_usd']:.2f} con Δln(exportaciones petroleras) "
      f"y {corr.loc['avg_price_usd_bbl','non_oil_exports_usd']:.2f} con Δln(no petroleras).")

## 2. Regresión log-log: exportaciones petroleras vs Brent

$\ln(X_t) = \alpha + \beta\,\ln(\text{Brent}_t) + \varepsilon_t$ — β es la elasticidad. Errores estándar robustos (HC1).

In [ ]:
m = df.dropna(subset=["oil_exports_usd", "avg_price_usd_bbl"])
X = sm.add_constant(np.log(m["avg_price_usd_bbl"])); y = np.log(m["oil_exports_usd"])
ols = sm.OLS(y, X).fit(cov_type="HC1"); print(ols.summary().tables[1])
b, lo, hi = ols.params.iloc[1], *ols.conf_int().iloc[1]
print(f"Conclusión: β = {b:.2f} (IC 95 % {lo:.2f} a {hi:.2f}), R² = {ols.rsquared:.2f}. "
      f"Un Brent 10 % más alto se asocia con ~{b*10:.1f} % más exportaciones petroleras ese año.")
sens = data["oil_sensitivity"].query("iso3 == 'ECU' and target == 'oil_exports'")
print("Chequeo vs vista SQL gold.vw_oil_sensitivity (años con 12 meses de precio):", sens[["elasticity_beta", "r_squared"]].round(3).to_dict("records"))

## 3. Diagnóstico de residuos

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(m.index, ols.resid, marker="o"); a1.axhline(0, color="grey"); a1.set_title("Residuos en el tiempo")
sm.qqplot(ols.resid, line="s", ax=a2); a2.set_title("QQ-plot"); plt.tight_layout(); plt.show()
from statsmodels.stats.stattools import durbin_watson
print(f"Conclusión: Durbin-Watson = {durbin_watson(ols.resid):.2f} (≈2 sin autocorrelación; <1,5 sugiere autocorrelación positiva).")

## 4. Proyección de exportaciones no petroleras (SARIMAX con tendencia)

Modelo: ln(no petroleras) con tendencia lineal y AR(1). Pocos puntos anuales → intervalos amplios.

In [ ]:
s = np.log(ecu.set_index("year")["non_oil_exports_usd"])
s.index = pd.PeriodIndex(s.index, freq="Y")
mod = SARIMAX(s, order=(1, 0, 0), trend="ct").fit(disp=False)
H = 5; fc = mod.get_forecast(H); ci = fc.conf_int(alpha=0.2)
yrs = [p.year for p in fc.predicted_mean.index]
fig, ax = plt.subplots()
ax.plot([p.year for p in s.index], np.exp(s) / 1e9, marker="o", label="Observado", color=COLORS["ECU"])
ax.plot(yrs, np.exp(fc.predicted_mean) / 1e9, ls="--", marker="o", color=COLORS["ECU"], label="Proyección")
ax.fill_between(yrs, np.exp(ci.iloc[:, 0]) / 1e9, np.exp(ci.iloc[:, 1]) / 1e9, alpha=0.15, color=COLORS["ECU"], label="IC 80 %")
ax.legend(); ax.set_title("Exportaciones no petroleras de Ecuador (USD mil millones)"); plt.show()
print(f"Conclusión: para {yrs[-1]} el modelo proyecta {np.exp(fc.predicted_mean.iloc[-1])/1e9:.1f} mil millones "
      f"(IC 80 %: {np.exp(ci.iloc[-1,0])/1e9:.1f}–{np.exp(ci.iloc[-1,1])/1e9:.1f}). AIC = {mod.aic:.1f}.")

## 5. Comparación con la versión ligera de la app

La app usa una tendencia log-lineal + AR(1) en numpy (sin statsmodels en la nube). Ambas deberían dar órdenes de magnitud similares.

In [ ]:
sys.path.insert(0, str(ROOT / "app")); import analytics
light = analytics.forecast_trend_ar1(ecu.year, ecu.non_oil_exports_usd, H)
light["sarimax_mean"] = np.exp(fc.predicted_mean.values)
display((light.set_index("year") / 1e9).round(2))

## Limitaciones

- ~25 observaciones anuales: poca potencia estadística.
- USD corrientes: mezclan precio y volumen.
- Comtrade revisa los últimos años; el pipeline incremental vuelve a pedirlos.
- No se modelan choques climáticos, sanitarios ni cambios regulatorios.